# 03 — Burned area and burn severity
## The August 2024 Galičica wildfire

**Case study:** a major wildfire on Galičica began on **5 August 2024** and burned for roughly two weeks.  
In this practical we map the spectral change using Sentinel-2 and **dNBR**.

By the end you will have:
- comparable pre-fire and post-fire composites;
- pre/post NBR;
- dNBR;
- a simple burn-severity classification;
- area statistics;
- an optional comparison with an EFFIS reference layer.

> **Important:** dNBR classes are not universal truth. Thresholds should be validated for vegetation type, season, sensor and management purpose.

### How to work with this notebook

- Run the **core** cells from top to bottom.
- Pause at the interpretation questions before moving on.
- Most code is provided; the task is to understand the workflow, change selected parameters, and defend the interpretation.
- If a live service fails, tell a trainer rather than spending the practical debugging infrastructure.
- Stretch tasks are optional and are intended for participants who finish the core workflow early.

## 1. Imports and Earth Engine

In [ ]:
from pathlib import Path
import os

import ee
import pandas as pd
import geopandas as gpd
import folium

GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "").strip()

def initialize_earth_engine():
    """Initialize Earth Engine using existing credentials or the normal auth flow."""
    try:
        if GEE_PROJECT_ID:
            ee.Initialize(project=GEE_PROJECT_ID)
        else:
            ee.Initialize()
    except Exception:
        print("Earth Engine authentication is required.")
        ee.Authenticate()
        try:
            if GEE_PROJECT_ID:
                ee.Initialize(project=GEE_PROJECT_ID)
            else:
                ee.Initialize()
        except Exception as exc:
            raise RuntimeError(
                "Earth Engine could not initialize. If your account requires a "
                "registered Google Cloud project, set the GEE_PROJECT_ID "
                "environment variable, restart the kernel, and run this cell again."
            ) from exc

initialize_earth_engine()
print("Earth Engine ready.")

## 2. Study area and dates

We use the same **canonical Galičica course AOI** as in Practical 01.

The fire was reported from early to mid-August 2024. We therefore compare:
- a **pre-fire** summer composite;
- a **post-fire** composite after the main event.

The windows are deliberately broad enough to obtain useful cloud-free observations.

In [ ]:
repo_root = Path.home() / "mystorage" / "fire-school"

aoi_candidates = [
    repo_root / "data" / "aoi" / "galicica_aoi.geojson",
    Path.cwd() / "data" / "aoi" / "galicica_aoi.geojson",
    Path.cwd().parent / "data" / "aoi" / "galicica_aoi.geojson",
]

AOI_PATH = next((p for p in aoi_candidates if p.exists()), None)
AOI_GDF = None

if AOI_PATH is not None:
    AOI_GDF = gpd.read_file(AOI_PATH).to_crs("EPSG:4326")
    aoi_geom = AOI_GDF.geometry.iloc[0]
    AOI = ee.Geometry(aoi_geom.__geo_interface__)
    centroid = aoi_geom.centroid
    CENTER = [centroid.y, centroid.x]
    print("Using canonical AOI:", AOI_PATH)
else:
    AOI = ee.Geometry.Rectangle([20.78, 40.86, 21.12, 41.18])
    CENTER = [41.02, 20.95]
    print("Canonical AOI file not found — using rectangular fallback.")

ZOOM = 10

PRE_START  = "2024-06-01"
PRE_END    = "2024-08-04"

POST_START = "2024-08-19"
POST_END   = "2024-09-30"

MAX_CLOUD = 50

print("AOI area (km²):", round(AOI.area().divide(1e6).getInfo(), 1))

## 3. Prepare Sentinel-2 composites

In [ ]:
def mask_s2_scl(img):
    scl = img.select("SCL")
    bad = (
        scl.eq(3)
        .Or(scl.eq(8))
        .Or(scl.eq(9))
        .Or(scl.eq(10))
        .Or(scl.eq(11))
    )

    # Keep only bands used below. This avoids auxiliary-band schema/order
    # differences across Sentinel-2 processing baselines.
    return (
        img.updateMask(bad.Not())
        .select(["B4", "B8", "B12"])
    )

def s2_composite(start_date, end_date):
    col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(AOI)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
        .map(mask_s2_scl)
    )
    print(start_date, "to", end_date, "scenes:", col.size().getInfo())
    return col.median().clip(AOI)

pre  = s2_composite(PRE_START, PRE_END)
post = s2_composite(POST_START, POST_END)

## 4. Inspect pre-fire and post-fire imagery

In [ ]:
def add_ee_layer(m, ee_image, vis_params, name):
    map_id = ee_image.getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
    ).add_to(m)

m = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

if AOI_GDF is not None:
    folium.GeoJson(
        AOI_GDF,
        name="Course AOI",
        style_function=lambda _: {
            "color": "black",
            "weight": 2,
            "fillOpacity": 0.0,
        },
    ).add_to(m)

vis_swir = {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3500}

add_ee_layer(m, pre, vis_swir, "Pre-fire SWIR/NIR/Red")
add_ee_layer(m, post, vis_swir, "Post-fire SWIR/NIR/Red")

folium.LayerControl().add_to(m)
m

### Before calculating anything

Switch repeatedly between the pre- and post-fire layers.

- Where do you see the clearest change?
- Is every change necessarily fire?
- Which confounders could affect a two-date comparison?

## 5. Calculate NBR and dNBR

In [ ]:
pre_nbr  = pre.normalizedDifference(["B8", "B12"]).rename("NBR_pre")
post_nbr = post.normalizedDifference(["B8", "B12"]).rename("NBR_post")

# Conventional sign: positive values generally indicate a drop in NBR after fire.
dnbr = pre_nbr.subtract(post_nbr).rename("dNBR")

print("dNBR ready.")

In [ ]:
m2 = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

add_ee_layer(
    m2, dnbr,
    {
        "min": -0.25,
        "max": 0.8,
        "palette": ["2166ac", "f7f7f7", "fdae61", "d73027", "7f0000"],
    },
    "dNBR"
)

folium.LayerControl().add_to(m2)
m2

## 6. Mask obvious water and built-up areas

Our training AOI includes parts of the Ohrid/Prespa surroundings.  
For a cleaner wildfire exercise we use **ESA WorldCover 2021** to exclude:
- water (`80`);
- built-up (`50`).

This is only a convenience mask, not a fire-validation layer.

In [ ]:
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
land_mask = worldcover.neq(80).And(worldcover.neq(50))

dnbr_land = dnbr.updateMask(land_mask)

## 7. Simple burn-severity classes

For teaching we start from commonly used dNBR ranges:

| dNBR | Training label |
|---:|---|
| < 0.10 | Unburned / very low change |
| 0.10–0.27 | Low |
| 0.27–0.44 | Moderate-low |
| 0.44–0.66 | Moderate-high |
| > 0.66 | High |

These thresholds are **starting points**, not universal ecological severity classes.

In [ ]:
severity = (
    ee.Image(0)
    .where(dnbr_land.gte(0.10).And(dnbr_land.lt(0.27)), 1)
    .where(dnbr_land.gte(0.27).And(dnbr_land.lt(0.44)), 2)
    .where(dnbr_land.gte(0.44).And(dnbr_land.lt(0.66)), 3)
    .where(dnbr_land.gte(0.66), 4)
    .updateMask(dnbr_land.mask())
    .rename("severity")
)

severity_names = {
    0: "Unburned / very low change",
    1: "Low",
    2: "Moderate-low",
    3: "Moderate-high",
    4: "High",
}

In [ ]:
m3 = folium.Map(location=CENTER, zoom_start=ZOOM, tiles="CartoDB positron")

add_ee_layer(
    m3, severity,
    {
        "min": 0,
        "max": 4,
        "palette": ["d9d9d9", "ffffb2", "fecc5c", "fd8d3c", "bd0026"],
    },
    "dNBR severity classes"
)

folium.LayerControl().add_to(m3)
m3

## 8. Calculate area by class

In [ ]:
area_image = ee.Image.pixelArea().divide(1e6).addBands(severity)

grouped = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
    geometry=AOI,
    scale=20,
    maxPixels=1e9,
    bestEffort=True,
).get("groups").getInfo()

area_df = pd.DataFrame(grouped).rename(columns={"sum": "area_km2"})
area_df["label"] = area_df["class"].map(severity_names)
area_df = area_df[["class", "label", "area_km2"]].sort_values("class")
area_df["area_km2"] = area_df["area_km2"].round(2)

area_df

### Don't over-interpret the table

An AOI-wide classification plus generic thresholds can classify non-fire changes as burned.  
The important questions are:

1. Does the mapped pattern agree with the known fire location?
2. Where does it disagree with reference data?
3. Why?

## 9. Compare with the EFFIS reference layer

The canonical Galičica EFFIS subset is already included in the repository:

`data/effis/Galicica.gpkg`

No separate download is required.

For this exercise we filter the archive to **August 2024** so that historical polygons from other years do not clutter the comparison.

In [ ]:
repo_root = Path.home() / "mystorage" / "fire-school"

effis_candidates = [
    repo_root / "data" / "effis" / "Galicica.gpkg",
    Path.cwd() / "data" / "effis" / "Galicica.gpkg",
    Path.cwd().parent / "data" / "effis" / "Galicica.gpkg",
]

EFFIS_PATH = next((p for p in effis_candidates if p.exists()), None)

if EFFIS_PATH is None:
    raise FileNotFoundError(
        "Canonical EFFIS file data/effis/Galicica.gpkg was not found. "
        "Run git pull in the fire-school repository."
    )

effis = gpd.read_file(EFFIS_PATH).to_crs("EPSG:4326").copy()
effis["FIREDATE"] = pd.to_datetime(effis["FIREDATE"], errors="coerce")
effis["FINALDATE"] = pd.to_datetime(effis["FINALDATE"], errors="coerce")
effis["AREA_HA"] = pd.to_numeric(effis["AREA_HA"], errors="coerce")

effis_aug2024 = effis[
    (effis["FIREDATE"] >= "2024-08-01") &
    (effis["FIREDATE"] <= "2024-08-18")
].copy()

print("EFFIS archive features:", len(effis))
print("August 2024 reference polygons:", len(effis_aug2024))

display(
    effis_aug2024[
        ["id", "FIREDATE", "FINALDATE", "COUNTRY", "COMMUNE", "AREA_HA"]
    ].sort_values(["FIREDATE", "AREA_HA"], ascending=[True, False])
)

target_effis = effis[effis["id"].astype(str) == "240575"].copy()
if not target_effis.empty:
    print(
        "Main Ohrid-side EFFIS reference polygon 240575:",
        float(target_effis.iloc[0]["AREA_HA"]),
        "ha",
    )

# Folium/GeoJSON cannot serialize pandas Timestamp objects directly.
effis_aug2024_map = effis_aug2024.copy()
target_effis_map = target_effis.copy()

for frame in [effis_aug2024_map, target_effis_map]:
    for col in ["FIREDATE", "FINALDATE"]:
        frame[col] = frame[col].dt.strftime("%Y-%m-%d")

m4 = folium.Map(location=[40.93, 20.84], zoom_start=11, tiles="CartoDB positron")

folium.GeoJson(
    effis_aug2024_map,
    name="EFFIS August 2024",
    tooltip=folium.GeoJsonTooltip(
        fields=["id", "FIREDATE", "COUNTRY", "COMMUNE", "AREA_HA"],
        aliases=["ID", "Start", "Country", "Commune", "Area (ha)"],
    ),
    style_function=lambda _: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.05,
    },
).add_to(m4)

if not target_effis_map.empty:
    folium.GeoJson(
        target_effis_map,
        name="EFFIS target 240575",
        style_function=lambda _: {
            "color": "#2c3e50",
            "weight": 3,
            "fillOpacity": 0.08,
        },
    ).add_to(m4)

add_ee_layer(
    m4,
    severity.updateMask(severity.gt(0)),
    {
        "min": 1,
        "max": 4,
        "palette": ["ffffb2", "fecc5c", "fd8d3c", "bd0026"],
    },
    "dNBR mapped change",
)

folium.LayerControl().add_to(m4)
display(m4)

## 10. Interpretation exercise — 15 minutes

In pairs, answer:

1. Where is the strongest mapped change?
2. Does the severity pattern look spatially coherent?
3. Find one place that may be a **false positive**.
4. How could topography, phenology, cloud masking, compositing dates or land-cover type affect dNBR?
5. If EFFIS is available, identify one disagreement between EFFIS and our map.
6. Which product would you trust more, and **for what specific purpose**?

There is no requirement that the two maps match perfectly.

## 11. Stretch tasks

Choose one if you finish early:

### A — Change the post-fire period
Try a later post-fire window. How stable is the mapped severity?

### B — Change thresholds
Move one dNBR threshold by ±0.05 and recompute areas.

### C — Compare with NDVI change
Calculate:

`NDVI_pre - NDVI_post`

Does it delineate the burn scar as clearly as dNBR?

### D — Add a second reference
Compare with an independent burned-area or fire-history product.

## 12. Output for the Galičica capstone

Save or note:

- your final dNBR map;
- area by severity class;
- **three defensible findings**;
- **one important limitation**;
- **one management-relevant interpretation**.

This practical feeds directly into the Friday burned-area/severity group task.